In [1]:
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
import h5py
import pandas as pd
import matplotlib.pyplot as plt

FS = 125

In [2]:
# STATISTICAL FEATURES

def mean_feature(signal):
    return np.mean(signal)

def std_feature(signal):
    return np.std(signal)

def skewness_feature(signal):
    return skew(signal)

def kurtosis_feature(signal):
    return kurtosis(signal)

def rms_feature(signal):
    return np.sqrt(np.mean(signal**2))

def max_feature(signal):
    return np.max(signal)

def min_feature(signal):
    return np.min(signal)

def range_feature(signal):
    return np.max(signal) - np.min(signal)

In [3]:
# PEAK-BASED FEATURES

def get_peaks(ppg):
    peaks, _ = find_peaks(
        ppg,
        distance=int(0.4 * FS)
    )
    return peaks

def heart_rate_feature(ppg):

    peaks = get_peaks(ppg)

    if len(peaks) < 2:
        return np.nan

    ppi = np.diff(peaks) / FS

    return 60 / np.mean(ppi)

def mean_peak_amplitude_feature(ppg):

    peaks = get_peaks(ppg)

    if len(peaks) == 0:
        return np.nan

    return np.mean(ppg[peaks])

def std_peak_amplitude_feature(ppg):

    peaks = get_peaks(ppg)

    if len(peaks) == 0:
        return np.nan

    return np.std(ppg[peaks])

def mean_peak_distance_feature(ppg):

    peaks = get_peaks(ppg)

    if len(peaks) < 2:
        return np.nan

    return np.mean(np.diff(peaks))




In [4]:
# VPG FEATURES

def max_vpg_feature(vpg):
    return np.max(vpg)

def min_vpg_feature(vpg):
    return np.min(vpg)

def mean_vpg_feature(vpg):
    return np.mean(vpg)

def std_vpg_feature(vpg):
    return np.std(vpg)

In [5]:
# APG FEATURES

def max_apg_feature(apg):
    return np.max(apg)

def min_apg_feature(apg):
    return np.min(apg)

def mean_apg_feature(apg):
    return np.mean(apg)

def std_apg_feature(apg):
    return np.std(apg)

In [6]:
# FREQUENCY DOMAIN FEATURES

def dominant_frequency_feature(ppg):

    fft_vals = np.abs(np.fft.rfft(ppg))

    freqs = np.fft.rfftfreq(
        len(ppg),
        d=1/FS
    )

    idx = np.argmax(fft_vals[1:]) + 1

    return freqs[idx]

def spectral_energy_feature(ppg):

    fft_vals = np.abs(np.fft.rfft(ppg))

    return np.sum(fft_vals**2)

In [7]:
def extract_features(ppg, vpg, apg):

    features = {

        # PPG statistics
        "mean_ppg": mean_feature(ppg),
        "std_ppg": std_feature(ppg),
        "skew_ppg": skewness_feature(ppg),
        "kurtosis_ppg": kurtosis_feature(ppg),
        "rms_ppg": rms_feature(ppg),
        "max_ppg": max_feature(ppg),
        "min_ppg": min_feature(ppg),
        "range_ppg": range_feature(ppg),

        # Peak features
        "heart_rate": heart_rate_feature(ppg),
        "mean_peak_amp": mean_peak_amplitude_feature(ppg),
        "std_peak_amp": std_peak_amplitude_feature(ppg),
        "mean_peak_distance": mean_peak_distance_feature(ppg),

        # VPG
        "max_vpg": max_vpg_feature(vpg),
        "min_vpg": min_vpg_feature(vpg),
        "mean_vpg": mean_vpg_feature(vpg),
        "std_vpg": std_vpg_feature(vpg),

        # APG
        "max_apg": max_apg_feature(apg),
        "min_apg": min_apg_feature(apg),
        "mean_apg": mean_apg_feature(apg),
        "std_apg": std_apg_feature(apg),

        # Frequency
        "dominant_freq": dominant_frequency_feature(ppg),
        "spectral_energy": spectral_energy_feature(ppg)
    }

    return features

In [9]:
import os

metadata = pd.read_excel("PPG-BP dataset.xlsx", header=1)

# Folder containing all txt files
signal_folder = "0_subject_clean"

rows = []

files = sorted(
    [f for f in os.listdir(signal_folder) if f.endswith(".txt")]
)

for file in files:
    name = os.path.splitext(file)[0]

    subject_id = int(name.split("_")[0])
    trial = int(name.split("_")[1])

    # -----------------------------
    # Read signal
    # -----------------------------
    ppg = np.loadtxt(os.path.join(signal_folder, file))

    # -----------------------------
    # Derivatives
    # -----------------------------
    vpg = np.gradient(ppg)
    apg = np.gradient(vpg)

    # -----------------------------
    # Extract features
    # -----------------------------
    features = extract_features(ppg, vpg, apg)

    # -----------------------------
    # Metadata row
    # -----------------------------
    info = metadata.loc[
        metadata["subject_ID"] == subject_id
    ]

    if info.empty:
        print(f"Metadata missing for subject {subject_id}")
        continue

    info = info.iloc[0].to_dict()

    # -----------------------------
    # Combine everything
    # -----------------------------
    row = {}

    row["subject_ID"] = subject_id
    row["trial"] = trial

    # demographic + BP information
    row.update(info)

    # extracted features
    row.update(features)

    rows.append(row)

# -----------------------------
# Final dataframe
# -----------------------------
final_df = pd.DataFrame(rows)

print(final_df.head())

print("Shape:", final_df.shape)

   subject_ID  trial  Num. Sex(M/F)  Age(year)  Height(cm)  Weight(kg)  \
0         100      1    69   Female         68         150          63   
1         100      2    69   Female         68         150          63   
2         100      3    69   Female         68         150          63   
3         103      1    70   Female         63         153          45   
4         103      2    70   Female         63         153          45   

   Systolic Blood Pressure(mmHg)  Diastolic Blood Pressure(mmHg)  \
0                            140                              82   
1                            140                              82   
2                            140                              82   
3                            120                              60   
4                            120                              60   

   Heart Rate(b/m)  ...    max_vpg    min_vpg  mean_vpg    std_vpg    max_apg  \
0               73  ...  36.363926 -20.139884  0.282051  11.44089

In [10]:
final_df.to_csv(
    "PPG_BP_with_features.csv",
    index=False
)

In [ ]:
# ppg = np.loadtxt("0_subject_clean/100_1.txt")

# print(np.mean(ppg))
# print(np.std(ppg))

9.841801804447284
123.56979341251964
